# Week 8 - Ensemble Learning: Pure NumPy Bagging and Boosting (SAMME)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vantage-Consortium-Co-Ltd/Ensemble-Learning/blob/phuriphat/notebooks/week8_colab_runner.ipynb)

## 📌 Project Overview
This interactive runner demonstrates pure **NumPy & CuPy** implementations of:
1. **BaggingClassifier**: Bootstrap Aggregation with parallel process pool execution, voting, and variance reduction.
2. **BoostingClassifier**: Multi-class Boosting using the **SAMME** (Stagewise Additive Modeling using a Multi-class Exponential loss function) algorithm and bias reduction.

Tested on the **Iris** benchmark dataset (`antoniopelusi/iris`) with an 80/20 stratified split, supporting both local CPU and NVIDIA A100 GPU acceleration on Google Colab.

### 🛠️ Cell 1: Colab Environment Setup & Hardware Acceleration
Detects runtime environment, checks for GPU acceleration (`nvidia-smi`), clones the repository if needed, and installs dependencies.

In [ ]:
import os
import sys
import subprocess

# Check if running inside Google Colab
is_colab = 'google.colab' in sys.modules or os.path.exists('/content')

if is_colab:
    print("🚀 Detected Google Colab environment.")
    print("🔍 Checking NVIDIA GPU Status (A100 / T4)...\n")
    !nvidia-smi
    
    print("\n📦 Installing dependencies...")
    !pip install --quiet datasets pyyaml scikit-learn pandas matplotlib tabulate joblib
    
    repo_path = "/content/Ensemble-Learning"
    if not os.path.exists(repo_path):
        print("📥 Cloning repository branch 'phuriphat'...")
        !git clone -b phuriphat https://github.com/Vantage-Consortium-Co-Ltd/Ensemble-Learning.git {repo_path}
    
    if f"{repo_path}/src" not in sys.path:
        sys.path.insert(0, f"{repo_path}/src")
    os.chdir(repo_path)
else:
    print("💻 Running in local development environment (CPU Mode).")
    from pathlib import Path
    current = Path(os.getcwd()).resolve()
    project_root = None
    for candidate in [current, current.parent, current.parent.parent]:
        if (candidate / "src" / "ensemble_learning").exists():
            project_root = candidate
            break
    if project_root is None:
        project_root = current
    os.chdir(str(project_root))
    repo_src = str(project_root / "src")
    if repo_src not in sys.path:
        sys.path.insert(0, repo_src)

# Verify compute backend
try:
    from ensemble_learning.utils.device import get_device_info
    info = get_device_info()
    print(f"⚡ Array Backend: {info['backend_module']} | Device: {info['device_name']} | CUDA Available: {info['cuda_available']}")
except Exception as e:
    import numpy as np
    print(f"⚡ NumPy fallback active ({np.__version__})")


### 📊 Cell 2: Dataset Ingestion & Stratified 80/20 Splitting
Loads the Iris dataset from Hugging Face Hub (`antoniopelusi/iris`) with caching and performs an 80/20 stratified split into Train (120 samples) and Test (30 samples).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

try:
    from ensemble_learning.data.make_dataset import load_iris_data, split_dataset
    df = load_iris_data(raw_data_dir="data/raw")
    X_train, X_test, y_train, y_test = split_dataset(df, train_ratio=0.8, test_ratio=0.2, random_state=42)
except Exception as exc:
    print(f"Import notice: {exc}. Using fallback loader.")
    from datasets import load_dataset
    try:
        ds = load_dataset("antoniopelusi/iris", split="train")
        df = pd.DataFrame(ds)
    except Exception:
        from sklearn.datasets import load_iris
        iris_raw = load_iris(as_frame=True)
        df = iris_raw.frame
        df.columns = ["sepal-length", "sepal-width", "petal-length", "petal-width", "target"]
    feature_cols = ["sepal-length", "sepal-width", "petal-length", "petal-width"]
    X = df[feature_cols].values
    y = df["target"].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"✅ Iris Dataset Loaded: {len(df)} total rows")
print(f"🎯 Train Split: {X_train.shape[0]} samples | Test Split: {X_test.shape[0]} samples")
print(f"📈 Class Distribution in Train: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"📉 Class Distribution in Test:  {dict(zip(*np.unique(y_test, return_counts=True)))}")
display(df.head())


### 🔍 Cell 3: Exploratory Data Analysis & Visualization
Examine feature distributions across species (`Setosa`, `Versicolor`, `Virginica`) and explore class separability via pairwise distributions.

In [ ]:
import matplotlib.pyplot as plt

species_labels = ["Iris-setosa", "Iris-versicolor", "Iris-virginica"]
colors = ["#1f77b4", "#2ca02c", "#d62728"]
feature_cols = ["sepal-length", "sepal-width", "petal-length", "petal-width"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for idx, (ax, col) in enumerate(zip(axes.flatten(), feature_cols)):
    for class_id, (sp_name, color) in enumerate(zip(species_labels, colors)):
        vals = df[df["target"] == class_id][col]
        ax.hist(vals, bins=12, alpha=0.6, label=sp_name, color=color, edgecolor="black")
    ax.set_title(f"Distribution: {col}", fontsize=11, fontweight="bold")
    ax.set_xlabel("cm")
    ax.set_ylabel("Frequency")
    ax.grid(alpha=0.3)
    if idx == 0:
        ax.legend()

plt.suptitle("Iris Feature Distributions by Species", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Petal Length vs. Petal Width Scatter
plt.figure(figsize=(8, 5))
for class_id, (sp_name, color) in enumerate(zip(species_labels, colors)):
    subset = df[df["target"] == class_id]
    plt.scatter(
        subset["petal-length"], subset["petal-width"],
        label=sp_name, color=color, s=60, edgecolors="black", alpha=0.85
    )
plt.title("Petal Dimensions: High Class Separability", fontsize=13, fontweight="bold")
plt.xlabel("Petal Length (cm)")
plt.ylabel("Petal Width (cm)")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


### 🧠 Cell 4: Pure NumPy Bagging and Boosting SAMME Architecture
We instantiate and evaluate our pure NumPy ensemble models:
- **`BaggingClassifier`**: Generates bootstrap samples $D_1, \dots, D_M$ with replacement, fits $M$ deep Decision Trees in parallel, and aggregates via majority voting.
- **`BoostingClassifier`**: Iteratively trains weak Decision Stumps (depth 1), calculating the multi-class SAMME estimator weight $\alpha_m = \log \frac{1 - \text{err}_m}{\text{err}_m} + \log(K - 1)$ and boosting hard-to-classify samples.

In [ ]:
from ensemble_learning.models.bagging import BaggingClassifier
from ensemble_learning.models.boosting import BoostingClassifier
from ensemble_learning.utils.metrics import compute_classification_metrics

# 1. Quick test: Bagging with 50 estimators
bag_model = BaggingClassifier(n_estimators=50, n_jobs=4, random_state=42)
bag_model.fit(X_train, y_train)
bag_preds = bag_model.predict(X_test)
bag_metrics = compute_classification_metrics(y_test, bag_preds)

# 2. Quick test: Boosting (SAMME) with 50 estimators
boost_model = BoostingClassifier(n_estimators=50, learning_rate=1.0, random_state=42)
boost_model.fit(X_train, y_train)
boost_preds = boost_model.predict(X_test)
boost_metrics = compute_classification_metrics(y_test, boost_preds)

print("=== Baseline Ensemble Performance (N=50) ===")
print(f"🌲 BaggingClassifier  -> Accuracy: {bag_metrics['accuracy']:.4f} | Macro F1: {bag_metrics['f1_macro']:.4f}")
print(f"🚀 BoostingClassifier -> Accuracy: {boost_metrics['accuracy']:.4f} | Macro F1: {boost_metrics['f1_macro']:.4f}")


### ⏱️ Cell 5: Benchmark Sweep across $N \in [10, 50, 100, 500]$
We benchmark both ensemble architectures across varying ensemble sizes $N \in [10, 50, 100, 500]$, measuring:
- **Fit Time (s)**: Training wall-clock time
- **Inference Latency (s)**: Prediction time on the test set
- **Accuracy & Macro F1 Score**

In [ ]:
from ensemble_learning.models.train import run_benchmark, format_benchmark_table

# Execute the benchmark sweep
sweep_estimators = [10, 50, 100, 500]
print(f"Running benchmark sweep across N = {sweep_estimators}...")
benchmark_results = run_benchmark(n_estimators_list=sweep_estimators)

# Display formatted ASCII table
print("\n" + format_benchmark_table(benchmark_results))

# Also convert to pandas DataFrame for interactive viewing
benchmark_df = pd.DataFrame(benchmark_results)
display(benchmark_df)


### 📈 Cell 6: Empirical Comparison Charts (Bagging vs. Boosting)
Visualizing Accuracy and Training Latency as functions of $N$ estimators:
- **Bagging (Variance Reduction)**: Averages independent trees to lower estimation variance without increasing bias. Performance stabilizes quickly as $N$ increases.
- **Boosting (Bias Reduction)**: Sequentially focuses on previously misclassified samples. While individual decision stumps have high bias, the linear combination yields low bias and sharp boundaries.

In [ ]:
bag_df = benchmark_df[benchmark_df["model"] == "Bagging"].sort_values("n_estimators")
boost_df = benchmark_df[benchmark_df["model"] == "Boosting"].sort_values("n_estimators")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Accuracy vs N
ax1.plot(bag_df["n_estimators"], bag_df["accuracy"], marker="o", linewidth=2.5, label="Bagging", color="#1f77b4")
ax1.plot(boost_df["n_estimators"], boost_df["accuracy"], marker="s", linewidth=2.5, label="Boosting (SAMME)", color="#d62728")
ax1.set_title("Test Accuracy vs. Number of Estimators (N)", fontsize=12, fontweight="bold")
ax1.set_xlabel("Number of Estimators (N)")
ax1.set_ylabel("Test Accuracy")
ax1.set_ylim([0.85, 1.02])
ax1.grid(True, alpha=0.3)
ax1.legend(loc="lower right")

# Subplot 2: Training Latency vs N
ax2.plot(bag_df["n_estimators"], bag_df["fit_time_sec"], marker="o", linewidth=2.5, label="Bagging (ProcessPool)", color="#1f77b4")
ax2.plot(boost_df["n_estimators"], boost_df["fit_time_sec"], marker="s", linewidth=2.5, label="Boosting (Sequential)", color="#d62728")
ax2.set_title("Training Latency vs. Number of Estimators (N)", fontsize=12, fontweight="bold")
ax2.set_xlabel("Number of Estimators (N)")
ax2.set_ylabel("Fit Time (seconds)")
ax2.grid(True, alpha=0.3)
ax2.legend(loc="upper left")

plt.tight_layout()
plt.show()


### 💾 Cell 7: Model Artifact Serialization & Export
Serializes the best trained models as `.pkl` artifacts and provides download buttons / Google Drive sync options for easy deployment.

In [ ]:
import joblib
from pathlib import Path
from ensemble_learning.models.predict import load_and_predict

models_dir = Path("models")
models_dir.mkdir(parents=True, exist_ok=True)

bagging_pkl = models_dir / "bagging_iris.pkl"
boosting_pkl = models_dir / "boosting_iris.pkl"

# Save models
joblib.dump(bag_model, bagging_pkl)
joblib.dump(boost_model, boosting_pkl)
print(f"💾 Serialized Bagging model to: {bagging_pkl}")
print(f"💾 Serialized Boosting model to: {boosting_pkl}")

# Verification with sample inference
sample_features = [5.1, 3.5, 1.4, 0.2]  # Expected: 0 (Iris-setosa)
pred_bag = load_and_predict(str(bagging_pkl), sample_features)
pred_boost = load_and_predict(str(boosting_pkl), sample_features)

species_names = ["Iris-setosa", "Iris-versicolor", "Iris-virginica"]
print(f"\n🔎 Verification Inference on {sample_features}:")
print(f"   - Bagging Prediction:  Class {pred_bag} ({species_names[pred_bag]})")
print(f"   - Boosting Prediction: Class {pred_boost} ({species_names[pred_boost]})")

# Download artifacts in Google Colab
if is_colab:
    try:
        from google.colab import files
        print("\n⬇️ Triggering browser downloads for serialized models...")
        files.download(str(bagging_pkl))
        files.download(str(boosting_pkl))
    except Exception as e:
        print(f"Google Colab download notice: {e}")
